Installing Ollama for every session:

In [2]:
!bash install.sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################### 100.0%##########                                   55.0%                                58.0%
>>> Adding ollama user to render group...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> Enabling and starting ollama service...
>>> NVIDIA GPU installed.


In [3]:
!ollama --version

]11;?\ollama version is 0.18.0


Importing Packages

In [4]:
import sys
# Ensure we aren't accidentally pulling from Roaming
sys.path = [p for p in sys.path if "Roaming" not in p]

try:
    from scipy import sparse
    import sklearn
    print("Success! Scipy Sparse and Sklearn are both loaded.")
except ImportError as e:
    print(f"Still failing: {e}")

Success! Scipy Sparse and Sklearn are both loaded.


In [5]:
import unsloth
import json
from datasets import Dataset
import sys
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer, SFTConfig , GRPOConfig, GRPOTrainer
import torch

import os
from dotenv import load_dotenv
load_dotenv()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


True

In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
#from langchain_community.chains import LLMChain
import json
from langchain_community.llms import Ollama
from langchain_community.chat_models import ChatOllama
import os
import urllib.request
from pathlib import Path
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
import pandas as pd
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings, OpenAIEmbeddings
#from langchain.evaluation import load_evaluator
from datasets import Dataset
from ragas.metrics.collections import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from ragas.llms import LangchainLLMWrapper
from datasets import load_dataset
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import Faithfulness, AnswerRelevancy, AnswerCorrectness, ContextUtilization, ContextPrecision
from openai import OpenAI
from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory
from ragas.metrics.base import Metric
from langchain_experimental.text_splitter import SemanticChunker

import random
from typing import List
from pydantic import BaseModel
from langchain_core.output_parsers import PydanticOutputParser, JsonOutputParser
from prompts import  RFT_EVAL_DATA_GEN_CONCISE

from pydantic import BaseModel, Field
from langchain_community.vectorstores import FAISS
#import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from IPython.display import display, Markdown
from transformers import TrainingArguments
from transformers import GenerationConfig

import gc
import re
load_dotenv()

/tmp/ipykernel_314140/1046198275.py:32: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, AnswerCorrectness, ContextUtilization, ContextPrecision
/tmp/ipykernel_314140/1046198275.py:32: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy, AnswerCorrectness, ContextUtilization, ContextPrecision
/tmp/ipykernel_314140/1046198275.py:32: DeprecationWarning: Importing AnswerCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerC

True

In [4]:
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
GPU: Tesla T4


In [5]:
import torch
print(f"PyTorch CUDA: {torch.version.cuda}")
print(torch.cuda.get_device_capability())
!nvcc --version

PyTorch CUDA: 12.8
(7, 5)


nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Tue_Oct_29_23:50:19_PDT_2024
Cuda compilation tools, release 12.6, V12.6.85
Build cuda_12.6.r12.6/compiler.35059454_0


In [1]:
!nvidia-smi

Mon Mar 16 05:00:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:1E.0 Off |                    0 |
| N/A   23C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!free -h

In [4]:
gsm8k_dataset = load_dataset('openai/gsm8k', 'main')
gsm8k_dataset_train = gsm8k_dataset['train']
gsm8k_dataset_train

Dataset({
    features: ['question', 'answer'],
    num_rows: 7473
})

In [5]:
gsm8k_dataset_test = gsm8k_dataset['test']
gsm8k_dataset_test

Dataset({
    features: ['question', 'answer'],
    num_rows: 1319
})

In [6]:
print("Checking the Question :")
display(Markdown(gsm8k_dataset_train[:1]['question'][0]))
print("Checking the Answer :")
display(Markdown(gsm8k_dataset_train[:1]['answer'][0]))

Checking the Question :


Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

Checking the Answer :


Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72

In [10]:
print("Checking the Question :")
display(Markdown(gsm8k_dataset_test[:1]['question'][0]))
print("Checking the Answer :")
display(Markdown(gsm8k_dataset_test[:1]['answer'][0]))

Checking the Question :


Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

Checking the Answer :


Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18

In [11]:
gsm8k_dataset_test[:2]['answer']

['Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\nShe makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.\n#### 18',
 'It takes 2/2=<<2/2=1>>1 bolt of white fiber\nSo the total amount of fabric is 2+1=<<2+1=3>>3 bolts of fabric\n#### 3']

In [6]:
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'
os.environ['UNSLOTH_STABLE_DOWNLOADS'] = '1'
os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"

lora_rank = 64

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B",
    max_seq_length = 2048, 
    load_in_4bit = True,  
    load_in_8bit = False, 
    full_finetuning = False, 
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.5, # Reduce if out of memory
)

==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-unsloth-bnb-4bit as a legacy tokenizer.


In [7]:
model = FastLanguageModel.get_peft_model(
    model=model,
    r=lora_rank,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha=lora_rank,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None
)
print("PEFT enabled LLAMA3.1:8b model ready..!")

Unsloth 2026.3.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


PEFT enabled LLAMA3.1:8b model ready..!


In [8]:
def llama_formatting(batch):

    prompt_list = []
    answer_list = []

    for question, answer in zip(batch["question"], batch["answer"]):

        if "####" in answer:
            thought_text, solution_text = answer.split("####", 1)
            solution_text = solution_text.strip()
        else:
            thought_text = answer
            solution_text = answer


        prompt = (
            "<|begin_of_text|>"
            "<|start_header_id|>system<|end_header_id|>\n"
            "You are a mathematical reasoning assistant. Solve the problem step by step. Return your response EXACTLY  in this format:" 
            "<think>"
            "step-by-step reasoning"
            "</think>"
            "<answer>"
            "final integer answer"
            "</answer>"
            "<|eot_id|>"
            "<|start_header_id|>user<|end_header_id|>\n"
            f"{question}\n"
            "<|eot_id|>"
            "<|start_header_id|>assistant<|end_header_id|>\n"
            f"<think>\n"
        )

        prompt_list.append(prompt)
        answer_list.append(solution_text)

    return {
        'prompt' : prompt_list,
        'answer' : answer_list
    }

In [9]:
train_dataset = gsm8k_dataset_train.map(llama_formatting,batched=True,load_from_cache_file=False)
print(train_dataset)
print("Train Dataset : ",train_dataset[90])
print("----------------------------------------------------------------------------")

gsm8k_dataset_sample = gsm8k_dataset_test.select(range(100))
test_dataset = gsm8k_dataset_sample.map(llama_formatting,batched=True,load_from_cache_file=False)
print(test_dataset)
print("Test Dataset : ",test_dataset[90])

Map: 100%|##########| 7473/7473 [00:00<?, ? examples/s]

Dataset({
    features: ['question', 'answer', 'prompt'],
    num_rows: 7473
})
Train Dataset :  {'question': 'Herman likes to feed the birds in December, January and February.  He feeds them 1/2 cup in the morning and 1/2 cup in the afternoon.  How many cups of food will he need for all three months?', 'answer': '90', 'prompt': '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\nYou are a mathematical reasoning assistant. Solve the problem step by step. Return your response EXACTLY  in this format:<think>step-by-step reasoning</think><answer>final integer answer</answer><|eot_id|><|start_header_id|>user<|end_header_id|>\nHerman likes to feed the birds in December, January and February.  He feeds them 1/2 cup in the morning and 1/2 cup in the afternoon.  How many cups of food will he need for all three months?\n<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n<think>\n'}
----------------------------------------------------------------------------


Map: 100%|##########| 100/100 [00:00<?, ? examples/s]

Dataset({
    features: ['question', 'answer', 'prompt'],
    num_rows: 100
})
Test Dataset :  {'question': 'Ted the T-Rex was planning to bring potato salad to the dinosaur picnic.  He knows that an adult dinosaur will eat 10 lbs of potato salad, and a child will eat half as much as an adult.  If there will be 20 adults and 5 children at the picnic, how many pounds of potato salad does Ted need to bring to the picnic if he hopes to have enough to feed everyone?', 'answer': '225', 'prompt': '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\nYou are a mathematical reasoning assistant. Solve the problem step by step. Return your response EXACTLY  in this format:<think>step-by-step reasoning</think><answer>final integer answer</answer><|eot_id|><|start_header_id|>user<|end_header_id|>\nTed the T-Rex was planning to bring potato salad to the dinosaur picnic.  He knows that an adult dinosaur will eat 10 lbs of potato salad, and a child will eat half as much as an adult.  If there

In [10]:
train_dataset[90]['prompt']

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\nYou are a mathematical reasoning assistant. Solve the problem step by step. Return your response EXACTLY  in this format:<think>step-by-step reasoning</think><answer>final integer answer</answer><|eot_id|><|start_header_id|>user<|end_header_id|>\nHerman likes to feed the birds in December, January and February.  He feeds them 1/2 cup in the morning and 1/2 cup in the afternoon.  How many cups of food will he need for all three months?\n<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n<think>\n'

In [18]:
train_dataset[90]['answer']

'90'

In [10]:
def normalize_answers(answer):

    if answer is None:
        return None

    answer = answer.lower().strip()
    answer = answer.replace(",","")

    answer = re.sub(r"[^0-9\.\-]","", answer)

    if answer.endswith(".0"):
        answer = answer[:-2]

    return answer

def extract_xml_answer(text):
    match = re.search(r"<answer>\s*(.*?)\s*</answer>", text, re.DOTALL)

    if match:
        return match.group(1).strip()

    return None

def truncate_answer(text):
    if "</answer>" in text:
        return text.split("</answer>")[0] + "</answer>"
    return text

In [16]:
#Reward Functions:

    
def correctness_reward_func(question, completions, answer, **kwargs) -> list[float]:
    #responses = [completion[0]['content'] for completion in completions]
    #q = question[0] #[-1]['content']
    reward = []

    truncated_responses = [truncate_answer(r) for r in completions]
    extracted_responses = [extract_xml_answer(r) for r in truncated_responses]
    
    #Normalize the responses and the answers:
    normalized_responses = [normalize_answers(r) for r in extracted_responses]
    #normalized_answers = [normalize_answers(r) for r in answer]

    normalized_answers = normalize_answers(answer[0])

    print('-'*20, f"Question:\n{question[0]}", f"\nAnswer:\n{answer[0]}", f"\nResponse:\n{completions[0]}", f"\nExtracted:\n{normalized_responses[0]}")

    for response in normalized_responses:

        if response is None:
            reward.append(-1.0)
        elif response == normalized_answers:
            reward.append(2.0)
        else:
            reward.append(-1.0)

    assert len(reward) == len(completions)        
    return reward

def int_reward_func(completions, **kwargs) -> list[float]:
    #responses = [completion[0]['content'] for completion in completions]
    reward = []

    truncated_responses = [truncate_answer(r) for r in completions]
    extracted_responses = [extract_xml_answer(r) for r in truncated_responses]

    #Normalize the responses and the answers:
    normalized_responses = [normalize_answers(r) for r in extracted_responses]

    for r in normalized_responses:
        if r is None:
            reward.append(-1.0)

        #re.fullmatch(r"-?\d+(\.\d+)?", r)
        elif re.fullmatch(r"-?\d+(\.\d+)?", r): #r.isdigit():
            reward.append(0.5)
        else:
            reward.append(0.0)

    assert (len(reward) == len(completions))     
    return reward

def strict_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    # pattern = r"^<think>\n.*?\n</think>\n<answer>\n.*?\n</answer>\n$"
    pattern = r"<think>.*?</think>\s*<answer>\s*\d+\s*</answer>"
    responses = [completion for completion in completions]
    matches = [re.search(pattern, r, re.DOTALL) for r in responses]

    reward = [0.3 if match else 0.0 for match in matches]
    assert (len(reward) == len(completions))
    return reward

def soft_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"<think>.*?</think>\s*<answer>.*?</answer>"
    responses = [completion for completion in completions]
    matches = [re.match(pattern, r) for r in responses]

    reward = [0.5 if match else 0.0 for match in matches]
    assert (len(reward) == len(completions))
    return reward

def count_xml(text) -> float:
    count = 0.0
    if text.count("<think>\n") == 1:
        count += 0.125
    if text.count("\n</think>\n") == 1:
        count += 0.125
    if text.count("\n<answer>\n") == 1:
        count += 0.125
        count -= len(text.split("\n</answer>\n")[-1])*0.001
    if text.count("\n</answer>") == 1:
        count += 0.125
        count -= (len(text.split("\n</answer>")[-1]) - 1)*0.001
    return count

def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    contents = [completion for completion in completions]
    reward = [count_xml(c) for c in contents]
    assert (len(reward) == len(completions))
    return reward

def length_penalty(completions, **kwargs):
    #reward = [-0.001 * len(c) for c in completions]
    reward = []
    for c in completions:
        if len(c) < 20:
            reward.append(-0.3) # higher penalty
        else:
            reward.append(0)

    assert len(reward) == len(completions)
    return reward

def end_tag_reward(completions, **kwargs):

    rewards = []

    for c in completions:

        if "</answer>" in c:
            after = c.split("</answer>")[-1]

            if after.strip() == "":
                rewards.append(0.2)
            else:
                rewards.append(-0.5)   # penalty
        else:
            rewards.append(-0.2) # penalty

    assert len(rewards) == len(completions)
    return rewards

def reasoning_nonempty_reward(completions, **kwargs):
    rewards = []
    for c in completions:
        if "<answer>" in c and "</think>" in c:
            reasoning = c.split("</think>")[0].strip()
            rewards.append(0.1 if len(reasoning) > 10 else -0.1)
        else:
            rewards.append(-0.1)
            
    assert len(rewards) == len(completions)
    return rewards

In [17]:

training_args = GRPOConfig(
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_steps = 55,
    # eval_steps=5,
    # eval_strategy="steps",
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 1,
    bf16 = is_bfloat16_supported(),
    fp16 = not is_bfloat16_supported(),
    per_device_train_batch_size = 2,
    #per_device_eval_batch_size=8,
    gradient_accumulation_steps = 8, # Increase to 4 for smoother training
    num_generations = 8, # Decrease if out of memory
    max_prompt_length = 256,
    max_completion_length = 256,
    # num_train_epochs = 1, # Set to 1 for a full training run
    max_steps = 400,
    save_steps = 400,
    max_grad_norm = 0.1,
    report_to = "wandb", # Can use Weights & Biases
    run_name="RFT_Llama3.1-8bn_v1_v2",
    output_dir = "llama31-8bn_RFT_v2",
    top_p = 0.95,
    temperature = 0.6
)

In [18]:
generation_config = GenerationConfig(
    max_new_tokens=256,
    temperature=0.5,
    top_p=0.95,
    stop_strings=["</answer>"]   # important
)

trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        #xmlcount_reward_func,
        # soft_format_reward_func,
        strict_format_reward_func,
        int_reward_func,
        correctness_reward_func,
        length_penalty,
        end_tag_reward,
        reasoning_nonempty_reward
    ],
    args = training_args,
    train_dataset = train_dataset,
    #generation_config=generation_config
    #eval_dataset=test_dataset,
    # generation_kwargs={
    #     "temperature": 0.5,
    #     "top_p": 0.90
    # },
)

In [15]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,473 | Num Epochs = 1 | Total steps = 300
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 167,772,160 of 8,198,033,408 (2.05% trained)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /teamspace/studios/this_studio/.netrc.
wandb: Currently logged in as: namrata-thakur5790 (namrata-thakur5790-ibm) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, instructor, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/
Passing `generation_config` together with generation-related arguments=({'disable_compile', 'cache_implementation', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
--- Logging error ---
Traceback (most recent call last):
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.11/logging/__init__.py", line 1110, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.11/logging/__init__.py", line 953, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/home/zeus/miniconda3/env

-------------------- Question:
A concert ticket costs $40. Mr. Benson bought 12 tickets and received a 5% discount for every ticket bought that exceeds 10. How much did Mr. Benson pay in all? 
Answer:
476 
Response:
</think><answer> 400</answer> 
Extracted:
400


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,sampling / sampling_logp_difference / mean,sampling / sampling_logp_difference / max,sampling / importance_sampling_ratio / min,sampling / importance_sampling_ratio / mean,sampling / importance_sampling_ratio / max,kl,rewards / strict_format_reward_func / mean,rewards / strict_format_reward_func / std,rewards / int_reward_func / mean,rewards / int_reward_func / std,rewards / correctness_reward_func / mean,rewards / correctness_reward_func / std,rewards / length_penalty / mean,rewards / length_penalty / std,rewards / end_tag_reward / mean,rewards / end_tag_reward / std
1,0.227458,0.208563,0.733004,105.562500,6.000000,256.000000,0.125000,84.071434,6.000000,242.000000,0,0,0,0,0,0.000013,0.000000,0.000000,0.343750,0.239357,0.250000,0.683130,-0.328938,0.282714,-0.056250,0.318264
2,0.060413,0.287500,0.680908,132.437500,41.000000,256.000000,0.125000,114.785721,41.000000,206.000000,No Log,No Log,No Log,No Log,No Log,0.000009,0.000000,0.000000,0.437500,0.170783,0.125000,0.500000,-0.406250,0.228757,0.131250,0.181544
3,0.112216,0.525937,0.730426,92.062500,34.000000,246.000000,0.000000,92.062500,34.000000,246.000000,No Log,No Log,No Log,No Log,No Log,0.000012,0.000000,0.000000,0.468750,0.125000,0.375000,0.806226,-0.299062,0.213954,-0.018750,0.335099
4,0.207314,-0.129313,0.629112,142.500000,33.000000,256.000000,0.312500,90.909096,33.000000,223.000000,No Log,No Log,No Log,No Log,No Log,0.000010,0.000000,0.000000,0.375000,0.223607,0.000000,0.000000,-0.479313,0.382385,-0.025000,0.295522
5,0.008529,-0.094625,0.403630,122.062500,13.000000,256.000000,0.125000,102.928574,13.000000,245.000000,No Log,No Log,No Log,No Log,No Log,0.000011,0.000000,0.000000,0.437500,0.170783,0.000000,0.000000,-0.413375,0.241021,-0.118750,0.350654
6,-0.098261,0.659375,0.849764,122.500000,13.000000,256.000000,0.125000,103.428574,13.000000,170.000000,No Log,No Log,No Log,No Log,No Log,0.000009,0.018750,0.075000,0.406250,0.201556,0.500000,0.894427,-0.309375,0.143046,0.043750,0.278014
7,0.097191,-0.067000,0.712036,123.687500,66.000000,256.000000,0.125000,104.785721,66.000000,233.000000,No Log,No Log,No Log,No Log,No Log,0.000010,0.000000,0.000000,0.343750,0.239357,0.125000,0.500000,-0.404500,0.239959,-0.131250,0.341992
8,0.241647,0.084625,0.748407,154.750000,14.000000,256.000000,0.250000,121.000000,14.000000,252.000000,No Log,No Log,No Log,No Log,No Log,0.000009,0.000000,0.000000,0.375000,0.223607,0.125000,0.500000,-0.459125,0.265599,0.043750,0.278014
9,0.125359,0.020875,0.688132,176.187500,52.000000,256.000000,0.250000,149.583344,52.000000,243.000000,No Log,No Log,No Log,No Log,No Log,0.000015,0.000000,0.000000,0.406250,0.201556,0.125000,0.500000,-0.541625,0.204275,0.031250,0.275000
10,0.249927,0.051188,0.532110,129.250000,58.000000,256.000000,0.062500,120.800003,58.000000,215.000000,No Log,No Log,No Log,No Log,No Log,0.000015,0.000000,0.000000,0.468750,0.125000,0.125000,0.500000,-0.467562,0.293293,-0.075000,0.343511


-------------------- Question:
Janet pays $40/hour for 3 hours per week of clarinet lessons and $28/hour for 5 hours a week of piano lessons. How much more does she spend on piano lessons than clarinet lessons in a year? 
Answer:
1040 
Response:
Janet pays $40/hour for 3 hours per week of clarinet lessons and $28/hour for 5 hours a week of piano lessons. How much more does she spend on piano lessons than clarinet lessons in a year?
</think><answer>360</answer> 
Extracted:
360
-------------------- Question:
Over the past five years, on July 4th, the high temperature for Washington, DC has been: 90 degrees in 2020, 90 degrees in 2019, 90 degrees in 2018, 79 degrees in 2017 and 71 degrees in 2016. What is the average temperature for July 4th in Washington, DC over the past 5 years? 
Answer:
84 
Response:
1. Sum the temperatures: 90 + 90 + 90 + 79 + 71 = 420
2. Divide by 5: 420 / 5 = 84
3. Round to the nearest whole number: 84.0
</think><answer>84</answer>
 
Extracted:
84
-----------------

profiling/Time taken: UnslothGRPOTrainer._calculate_rewards,▂▂▃▂▂▂▂▂▂▂▁▂█▁▂▅▇▅▃▄▁▂▄▅▅▃▁▆▃▅▄▅▄▅▄▁▁▃▆▆
profiling/Time taken: UnslothGRPOTrainer._prepare_inputs,▁▁▁▁▁▁▁▁▁█▄▁▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▂▁▁
profiling/Time taken: UnslothGRPOTrainer.correctness_reward_func,▄▆▄▄▅▄▃▆▄▄▃▄▄▅█▅▇▁▇▄▄▅▃▇▅▅▆▇▆▅▇▄▇▄▆▃▄▂▃▄
profiling/Time taken: UnslothGRPOTrainer.end_tag_reward,▃▅▆▇▄▆▃▄▇▄▃▄▆▆▅▄▄▅▆▄▄▅▅▃▆▄▃▄▄▄▄▆▄▁▂▄▆▃▃█
profiling/Time taken: UnslothGRPOTrainer.int_reward_func,▂▆▇▂▆▃▃▃▅▂█▄▇▅▅▇▃▄▄▅▁▃▅▃▃▄▅▄▅▄▄▅▅▅▇▃▁▅▄▅
profiling/Time taken: UnslothGRPOTrainer.length_penalty,▂▁▄▃▁▂▁▃▂▃▃▂▁▃▁▅▃▄▃▄█▃▆▃▃▃▃▃▃▃▃▂▂▃▄▃▃▂▁▄
profiling/Time taken: UnslothGRPOTrainer.strict_format_reward_func,▂▂▂▂▂▂▂▁▁▃▂▂▁▂▃▆▄▅▃▅▄▃▄▁▅▄█▂▄▄▂▄▄▅▂▄▁▃▅▅
profiling/Time taken: UnslothGRPOTrainer.transformers.generate,██▇██▆█▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/clip_ratio/high_max,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/clip_ratio/high_mean,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+31,...


TrainOutput(global_step=300, training_loss=0.03687225305038737, metrics={'train_runtime': 12732.4543, 'train_samples_per_second': 0.377, 'train_steps_per_second': 0.024, 'total_flos': 0.0, 'train_loss': 0.03687225305038737})

V2

In [19]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,473 | Num Epochs = 1 | Total steps = 400
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 167,772,160 of 8,198,033,408 (2.05% trained)


-------------------- Question:
A concert ticket costs $40. Mr. Benson bought 12 tickets and received a 5% discount for every ticket bought that exceeds 10. How much did Mr. Benson pay in all? 
Answer:
476 
Response:
</think><answer> 400</answer> cling
 
Extracted:
400


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,sampling / sampling_logp_difference / mean,sampling / sampling_logp_difference / max,sampling / importance_sampling_ratio / min,sampling / importance_sampling_ratio / mean,sampling / importance_sampling_ratio / max,kl,rewards / strict_format_reward_func / mean,rewards / strict_format_reward_func / std,rewards / int_reward_func / mean,rewards / int_reward_func / std,rewards / correctness_reward_func / mean,rewards / correctness_reward_func / std,rewards / length_penalty / mean,rewards / length_penalty / std,rewards / end_tag_reward / mean,rewards / end_tag_reward / std,rewards / reasoning_nonempty_reward / mean,rewards / reasoning_nonempty_reward / std
1,0.056933,-0.850000,1.228573,125.750000,6.000000,256.000000,0.250000,82.333336,6.000000,246.000000,0,0,0,0,0,0.000010,0.000000,0.000000,0.062500,0.655108,-0.812500,0.750000,-0.018750,0.075000,-0.118750,0.312450,0.037500,0.095743
2,0.118922,-0.656250,1.163904,168.937500,41.000000,256.000000,0.250000,139.916672,41.000000,256.000000,No Log,No Log,No Log,No Log,No Log,0.000008,0.000000,0.000000,0.125000,0.670820,-0.812500,0.750000,0.000000,0.000000,-0.031250,0.289180,0.062500,0.080623
3,0.071142,0.056250,1.232294,101.375000,34.000000,256.000000,0.062500,91.066673,34.000000,181.000000,No Log,No Log,No Log,No Log,No Log,0.000008,0.000000,0.000000,0.406250,0.375000,-0.437500,1.209339,0.000000,0.000000,-0.000000,0.314113,0.087500,0.050000
4,0.322424,-0.756250,0.733516,140.812500,23.000000,256.000000,0.312500,88.454552,23.000000,164.000000,No Log,No Log,No Log,No Log,No Log,0.000010,0.000000,0.000000,0.281250,0.515388,-1.000000,0.000000,0.000000,0.000000,-0.112500,0.336403,0.075000,0.068313
5,0.054191,-0.943750,0.914914,144.375000,11.000000,256.000000,0.312500,93.636368,11.000000,202.000000,No Log,No Log,No Log,No Log,No Log,0.000008,0.000000,0.000000,0.093750,0.663796,-1.000000,0.000000,0.000000,0.000000,-0.075000,0.304412,0.037500,0.095743
6,-0.034370,0.256250,1.251395,107.500000,18.000000,256.000000,0.125000,86.285721,18.000000,153.000000,No Log,No Log,No Log,No Log,No Log,0.000010,0.000000,0.000000,0.312500,0.512348,-0.062500,1.436141,0.000000,0.000000,-0.068750,0.328063,0.075000,0.068313
7,0.029077,-0.393750,1.515281,145.187500,14.000000,256.000000,0.250000,108.250000,14.000000,234.000000,No Log,No Log,No Log,No Log,No Log,0.000008,0.000000,0.000000,0.093750,0.663796,-0.437500,1.209339,0.000000,0.000000,-0.075000,0.304412,0.025000,0.100000
8,-0.102940,0.118750,0.853464,101.500000,18.000000,200.000000,0.000000,101.500000,18.000000,200.000000,No Log,No Log,No Log,No Log,No Log,0.000008,0.000000,0.000000,0.500000,0.000000,-0.625000,1.024695,0.000000,0.000000,0.156250,0.175000,0.087500,0.050000
9,-0.190515,-0.543750,1.043019,150.000000,23.000000,256.000000,0.187500,125.538467,23.000000,249.000000,No Log,No Log,No Log,No Log,No Log,0.000005,0.000000,0.000000,0.218750,0.515388,-0.812500,0.750000,0.000000,0.000000,-0.025000,0.313050,0.075000,0.068313
10,0.038231,-0.856250,0.452438,130.000000,12.000000,256.000000,0.250000,88.000000,12.000000,221.000000,No Log,No Log,No Log,No Log,No Log,0.000011,0.000000,0.000000,0.125000,0.591608,-1.000000,0.000000,0.000000,0.000000,-0.006250,0.290904,0.025000,0.100000


-------------------- Question:
Janet pays $40/hour for 3 hours per week of clarinet lessons and $28/hour for 5 hours a week of piano lessons. How much more does she spend on piano lessons than clarinet lessons in a year? 
Answer:
1040 
Response:
Janet pays $40/hour for 3 hours per week of clarinet lessons and $28/hour for 5 hours a week of piano lessons. How much more does she spend on piano lessons than clarinet lessons in a year?
If Janet pays $40/hour for 3 hours per week of clarinet lessons and $28/hour for 5 hours a week of piano lessons, we can find the amount she spends on clarinet lessons in one year by multiplying the hourly rate by the number of hours per week by the number of weeks in a year. In this case, the number of hours per week is 3 and the number of weeks in a year is 52, so Janet spends $3,120 per year on clarinet lessons.
The amount she spends on piano lessons in one year is found in the same way, except that we multiply the hourly rate by the number of hours per w

profiling/Time taken: UnslothGRPOTrainer._calculate_rewards,▂▃▂▂▃▂▃▂▁▅▁▂▂▁▂▂▁▁▂▄▃▇▃▂▂▂▂▂▃▃▁▁▂▂█▂▂▂▃▁
profiling/Time taken: UnslothGRPOTrainer._prepare_inputs,▁▁█▁▁▁▁▁▁▁▁▄▁▁▁▁▁▁▁▁▁▁▅▄▁▁▁▁▁█▁▁▁▁▁▁▅▇▄▁
profiling/Time taken: UnslothGRPOTrainer.correctness_reward_func,▂▅▆▄█▃▃▆▂▃▄▅▂▅▄▂▅▄█▂▃▆▃▃▃▃▄▄▃▂▃▃▃▃▁▃▃▇▁▂
profiling/Time taken: UnslothGRPOTrainer.end_tag_reward,▃▆▅▄▄▅▆▂▂▁▃▆▃▅▆▅▄█▅▃▃▃▂▁▃▂▆▅▆▂▄▃▅▅▄█▂▃▆▃
profiling/Time taken: UnslothGRPOTrainer.int_reward_func,▇▁▆▄▅▄▇▆▂▅▃▃▃▄▃▆▆▃▅▄▄▅▃▃▅▇▄▅█▄▄▆▃▅▄▄▄▄▃▃
profiling/Time taken: UnslothGRPOTrainer.length_penalty,▄▃▄▃▂▁▃▂▄▃▃▁▃▂▃▄█▃▂▂▃▂▃▃▃▂▃▂▃▂▄▃▂▂▃▃▃▃▃▅
profiling/Time taken: UnslothGRPOTrainer.reasoning_nonempty_reward,▄▄▇▃▅▅▄▅▇▅▄▄▃▇▃▇▄▄▁▃▃▃▃▆▄▃▅▇▄▄▇█▆▅▅▃▃▄▂▆
profiling/Time taken: UnslothGRPOTrainer.strict_format_reward_func,▁▁▁▄▂▁▂▁▂▇▂▂▂▂▁▂▂▁▄▂▂▃█▁▃▁▄▂▁▂▂▂▂▁▁▂▅▁▁▁
profiling/Time taken: UnslothGRPOTrainer.transformers.generate,█▇█▇█▆█▁█▃▁█▂▂▃▇▅▅▃▄█▄█▂▇▆▄▆▅▅█▇▃▆▁▆▃▄█▂
train/clip_ratio/high_max,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+34,...


TrainOutput(global_step=400, training_loss=0.01986133234595627, metrics={'train_runtime': 36691.3141, 'train_samples_per_second': 0.174, 'train_steps_per_second': 0.011, 'total_flos': 0.0, 'train_loss': 0.01986133234595627})

In [20]:
#This is merge LoRA adapters with model weights before pushing to Huggingface Hub:
model.push_to_hub_merged("NamrataThakur/llama31-8bn_Reinforcement-Fine-Tuned", tokenizer, 
                         save_method = "merged_16bit", token = os.environ["HF_TOKEN"])

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Found HuggingFace hub cache directory: /teamspace/studios/this_studio/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [05:19<00:00, 79.95s/it]


Unsloth: Merge process complete. Saved to `/teamspace/studios/this_studio/Fine-tuning-LLMs-Strategies/notebooks/NamrataThakur/llama31-8bn_Reinforcement-Fine-Tuned`


In [21]:
#This is merge LoRA adapters with model weights before saving in the local directory:
model.save_pretrained_merged("llama31-8bn_RFT", tokenizer, save_method = "merged_16bit")

Found HuggingFace hub cache directory: /teamspace/studios/this_studio/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [00:55<00:00, 13.82s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [01:20<00:00, 20.00s/it]


Unsloth: Merge process complete. Saved to `/teamspace/studios/this_studio/Fine-tuning-LLMs-Strategies/notebooks/llama31-8bn_RFT`


In [23]:
#Creating the GGUF file
!python llama.cpp/convert_hf_to_gguf.py ./llama31-8bn_RFT --outfile llama31-8bn_RFT_f16.gguf --outtype f16

INFO:hf-to-gguf:Loading model: llama31-8bn_RFT
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00004.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00004.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00003-of-00004.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00004-of-00004.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,           torch.bfloat16 --> F16, shape = {4096, 128256}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.bfloat16 --> F32, shape = {4096}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.bfloat16 --> F16, shape = {14336, 4096}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.bfloat16 --> F16, shape = {4096, 14336}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         to

In [22]:
model.push_to_hub_gguf(
    "NamrataThakur/llama31-8bn_Reinforcement-Fine-Tuned", 
    tokenizer,
    quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
    token = os.environ["HF_TOKEN"], 
)

Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /teamspace/studios/this_studio/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


Uploading config.json...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/NamrataThakur/llama31-8bn_Reinforcement-Fine-Tuned
Unsloth: Cleaning up temporary files...


'NamrataThakur/llama31-8bn_Reinforcement-Fine-Tuned'

In [ ]:
# Command: ollama create <model_name> -f Modelfile -q <quantised_version>
!ollama create llama31-8bn-RFT -f Modelfile_llamaRFT -q f16

In [ ]:
!ollama list

In [7]:
!ollama pull hf.co/NamrataThakur/llama31-8bn_Reinforcement-Fine-Tuned:Q8_0

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling f4b58543fc0e: 100% ▕██████████████████▏ 8.5 GB                         
pulling 3f71e9d0b371: 100% ▕██████████████████▏  404 B                         
verifying sha256 digest 
writing manifest 
success 


In [8]:
#Check the fine-tuned model name in "ollama list".
ft_model = ChatOllama(
    model="hf.co/NamrataThakur/llama31-8bn_Reinforcement-Fine-Tuned:Q8_0", #Give the same model name that is used in "ollama create" command
    temperature=0.01,
    num_predict=1024,      
    repeat_penalty=1.2,
)


/tmp/ipykernel_314140/3240320075.py:2: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  ft_model = ChatOllama(


In [6]:
openai_api_key = os.getenv("OPENAI_API_KEY")

In [29]:
smallThoughts_dataset = load_dataset("SmallDoge/SmallThoughts")
smallThoughts_eval_dataset = smallThoughts_dataset['test']
smallThoughts_eval_dataset

Dataset({
    features: ['problem', 'solution', 'messages', 'system_prompt'],
    num_rows: 2000
})

In [37]:
def smallThoughts_formatting(batch):

    formatted_data = []

    for question, answer in zip(batch["problem"], batch["solution"]):

        text = (
            "<|begin_of_text|>"
            "<|start_header_id|>system<|end_header_id|>\n"
            "You are a mathematical reasoning assistant. Solve the problem step by step. Return your response EXACTLY  in this format:\n" 
            "<think>"
            "step-by-step reasoning"
            "</think>"
            "<answer>"
            "final answer"
            "</answer>"
            "<|eot_id|>"
            "<|start_header_id|>user<|end_header_id|>\n"
            f"{question}\n"
            "<|eot_id|>"
            "<|start_header_id|>assistant<|end_header_id|>\n"

        )

        formatted_data.append(text )

    return {"text": formatted_data}

In [38]:
smallThoughts_eval_dataset_formatted = smallThoughts_eval_dataset.map(smallThoughts_formatting,batched=True)
smallThoughts_eval_dataset_formatted

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset({
    features: ['problem', 'solution', 'messages', 'system_prompt', 'text'],
    num_rows: 2000
})

In [39]:
smallThoughts_eval_dataset_formatted['text'][0]

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\nYou are a mathematical reasoning assistant. Solve the problem step by step. Return your response EXACTLY  in this format:\n<think>step-by-step reasoning</think><answer>final answer</answer><|eot_id|><|start_header_id|>user<|end_header_id|>\nReturn your final response within \\boxed{}. Given the function \\( y = f(x) \\) which has an inverse function \\( y = f^{-1}(x) \\), what is the equation of the curve obtained after rotating the graph of \\( y = f(x) \\) counterclockwise by \\( 90^{\\circ} \\) around the point \\( (1, -1) \\)?\n\n(A) \\( y = f^{-1}(-x) - 2 \\)  \n(B) \\( y = -f^{-1}(-x) - 2 \\)  \n(C) \\( y = f^{-1}(-x + 1) - 1 \\)  \n(D) \\( y = f^{-1}(-x - 1) + 1 \\)\n<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n'

In [40]:
class RFTEvalDataset(BaseModel):
    
    thought: str = Field(description="The detailed thinking process and complexity analysis")
    solution: str = Field(description="The step-by-step final mathematical solution")

In [42]:
def custom_parser(text: str) -> RFTEvalDataset:
    """Manually extracts tags since with_structured_output is missing."""
    thought_match = re.search(r'<think>(.*?)</think>', text, re.DOTALL)
    solution_match = re.search(r'(?i)<answer\s*>(.*?)</answer\s*>', text, re.DOTALL)
    
    return RFTEvalDataset(
        thought=thought_match.group(1).strip() if thought_match else "Parsing Error",
        solution=solution_match.group(1).strip() if solution_match else text # Fallback to raw text
    )

def generate_answers(dataset, sample_size = 10):
    
    rft_eval_data = []

    prompt = PromptTemplate(
                            template=RFT_EVAL_DATA_GEN_CONCISE,
                            input_variables = ["problem"]
                            )
    
    #structured_llm = ft_model.with_structured_output(RFTEvalDataset)
    rft_chain = prompt | ft_model

    #Selecting 20 rows for evaluation:
    data = dataset.select(range(sample_size))
    print(f"Total Evaluation Samples : {len(data)}")

    for idx, row in enumerate(data):
    
        try:

            print(f"Processing Sample {idx+1}. Problem length: {len(row['problem'])}")
            
            res = rft_chain.invoke(input={'problem':row['text']})
            print(res)

            structured_output = custom_parser(res.content)

            rft_eval_data.append({
                "question" : row['problem'],
                "contexts" : [row['solution']],
                "ground_truth" : row['solution'],
                "thought" : structured_output.thought,
                "answer" : structured_output.solution,
            })

            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            print(f"Generation Completed for Sample : {idx+1}")
            print("==========================================================")
        
        except Exception as e:
            print(f"FAILED at Sample {idx+1}: {str(e)}")
            continue 

    #Saving the answer to csv:
    test_generated_dataset = pd.DataFrame.from_records(data=rft_eval_data)
    test_generated_dataset.to_csv("../data/testEvalData_llama32-8bn_RFT_v2.csv",index=False)

    #Converting to Huggingface Dataset format for RAGAS evaluation:
    rft_eval_data = Dataset.from_list(rft_eval_data)
    return rft_eval_data

In [43]:
rft_eval_data = generate_answers(smallThoughts_eval_dataset_formatted, sample_size = 50)
rft_eval_data

Total Evaluation Samples : 50
Processing Sample 1. Problem length: 415


content="    <think>\n        **Analysis:** The goal is to find the equation of a curve obtained after rotating the graph of $y=f(x)$ counterclockwise by $90^{\\circ}$ around point $(1,−1)$.\n        \n        **Approach:**\n        - Rotate the function about its center (i.e., translate it so that origin becomes new center).\n            - Translate x-coordinate to be 0 and y-coordinate to be −2. \n                - This is done by subtracting $x$ from each point in the graph of $y=f(x)$.\n                    - The equation will become: $$f(-x-1)$$\n        - Rotate it counterclockwise about origin (i.e., rotate around $(0, 0)$).\n            - Apply rotation formula:\n                - If a function is rotated by $\\theta$ degrees in the counter-clockwise direction then its new coordinates are given as follows: $$\\begin{bmatrix}x'\\\\y'\\end{bmatrix}= \\begin{bmatrix}\\cos(\\theta)&-\\sin(\\theta)\\\\\\sin(\\theta)&\\cos(\\theta)\\end{bmatrix}. \\begin{bmatrix} x \\\\ y \\end{bmatri

Dataset({
    features: ['question', 'contexts', 'ground_truth', 'thought', 'answer'],
    num_rows: 50
})

In [7]:
rft_base_eval_data = pd.read_csv("../data/testEvalData_llama32-8bn_RFT_v2.csv")
print(f"Shape of Eval Data : {rft_base_eval_data.shape}")
rft_base_eval_data.head()

Shape of Eval Data : (50, 5)


,question,contexts,ground_truth,thought,answer
0,Return your final response within \boxed{}. Gi...,['**Solution:**\nTo determine the equation of ...,**Solution:**\nTo determine the equation of th...,**Analysis:** The goal is to find the equation...,"Thus, the transformed curve has an equation of..."
1,Return your final response within \boxed{}. Pr...,['**Solution:**\n\nTo prove the inequality:\n\...,**Solution:**\n\nTo prove the inequality:\n\[\...,The problem is moderately complex. I will brea...,The goal is to prove that:\n\n$$\n(a x + b u)^...
2,Return your final response within \boxed{}. If...,['**Solution:**\nGiven the equation \\(\\log_2...,**Solution:**\nGiven the equation \(\log_2 x^2...,Analysis: The problem is moderately complex. I...,"Thus, the transformed equation is $x^{\frac{3}..."
3,Return your final response within \boxed{}. A ...,['**Solution:**\n1. The original ratio of butt...,**Solution:**\n1. The original ratio of butter...,**Analysis:** The problem requires us to find ...,"Thus, \boxed{200} mL of butter is required."
4,Return your final response within \boxed{}. Gi...,['**Solution:**\n\n1. **Separate the integer p...,**Solution:**\n\n1. **Separate the integer par...,The problem is moderately complex. I will brea...,"Thus, the transformed equation becomes $\frac{..."


In [8]:
rft_base_eval_data = rft_base_eval_data[rft_base_eval_data['answer'].notnull()]
rft_base_eval_data.shape

(48, 5)

In [9]:
def parse_answers(text):
    # Added (?i) for case-insensitivity and \s* for potential spaces
    solution_match = re.search(r'(?i)<answer\s*>(.*?)</answer\s*>', text, re.DOTALL)
    
    if solution_match:
        return solution_match.group(1).strip()
    
    # If the tag wasn't closed properly, try to grab everything after the opening tag
    fallback_match = re.search(r'(?i)<answer\s*>(.*)', text, re.DOTALL)
    if fallback_match:
        return fallback_match.group(1).strip()
        
    return text

rft_base_eval_data['parsed_answer'] = rft_base_eval_data['answer'].apply(parse_answers)
rft_base_eval_data.head()

,question,contexts,ground_truth,thought,answer,parsed_answer
0,Return your final response within \boxed{}. Gi...,['**Solution:**\nTo determine the equation of ...,**Solution:**\nTo determine the equation of th...,**Analysis:** The goal is to find the equation...,"Thus, the transformed curve has an equation of...","Thus, the transformed curve has an equation of..."
1,Return your final response within \boxed{}. Pr...,['**Solution:**\n\nTo prove the inequality:\n\...,**Solution:**\n\nTo prove the inequality:\n\[\...,The problem is moderately complex. I will brea...,The goal is to prove that:\n\n$$\n(a x + b u)^...,The goal is to prove that:\n\n$$\n(a x + b u)^...
2,Return your final response within \boxed{}. If...,['**Solution:**\nGiven the equation \\(\\log_2...,**Solution:**\nGiven the equation \(\log_2 x^2...,Analysis: The problem is moderately complex. I...,"Thus, the transformed equation is $x^{\frac{3}...","Thus, the transformed equation is $x^{\frac{3}..."
3,Return your final response within \boxed{}. A ...,['**Solution:**\n1. The original ratio of butt...,**Solution:**\n1. The original ratio of butter...,**Analysis:** The problem requires us to find ...,"Thus, \boxed{200} mL of butter is required.","Thus, \boxed{200} mL of butter is required."
4,Return your final response within \boxed{}. Gi...,['**Solution:**\n\n1. **Separate the integer p...,**Solution:**\n\n1. **Separate the integer par...,The problem is moderately complex. I will brea...,"Thus, the transformed equation becomes $\frac{...","Thus, the transformed equation becomes $\frac{..."


In [7]:
display(Markdown(rft_base_eval_data.iloc[0]['question']))

Return your final response within \boxed{}. Given the function \( y = f(x) \) which has an inverse function \( y = f^{-1}(x) \), what is the equation of the curve obtained after rotating the graph of \( y = f(x) \) counterclockwise by \( 90^{\circ} \) around the point \( (1, -1) \)?

(A) \( y = f^{-1}(-x) - 2 \)  
(B) \( y = -f^{-1}(-x) - 2 \)  
(C) \( y = f^{-1}(-x + 1) - 1 \)  
(D) \( y = f^{-1}(-x - 1) + 1 \)

In [8]:
display(Markdown(rft_base_eval_data.iloc[0]['ground_truth']))

**Solution:**
To determine the equation of the curve after rotating \( y = f(x) \) 90° counterclockwise around \( (1, -1) \):

1. **Translate the point \( (t, f(t)) \) to the origin-centered system**:
   \[
   (t - 1, f(t) + 1)
   \]
2. **Apply 90° counterclockwise rotation** (transforms \( (a, b) \to (-b, a) \)):
   \[
   (- (f(t) + 1), t - 1)
   \]
3. **Translate back to the original coordinate system**:
   \[
   (-f(t) - 1 + 1, t - 1 - 1) = (-f(t), t - 2)
   \]
4. **Relabel the rotated coordinates as \( (x, y) \)**:
   \[
   x = -f(t) \quad \Rightarrow \quad t = f^{-1}(-x)
   \]
   Substitute \( t \) into \( y = t - 2 \):
   \[
   y = f^{-1}(-x) - 2
   \]

Thus, the transformed equation is \( y = f^{-1}(-x) - 2 \), making the correct answer \(\boxed{\text{A}}\).


In [10]:
display(Markdown(rft_base_eval_data.iloc[0]['thought']))

**Analysis:** The goal is to find the equation of a curve obtained after rotating the graph of $y=f(x)$ counterclockwise by $90^{\circ}$ around point $(1,−1)$.
        
        **Approach:**
        - Rotate the function about its center (i.e., translate it so that origin becomes new center).
            - Translate x-coordinate to be 0 and y-coordinate to be −2. 
                - This is done by subtracting $x$ from each point in the graph of $y=f(x)$.
                    - The equation will become: $$f(-x-1)$$
        - Rotate it counterclockwise about origin (i.e., rotate around $(0, 0)$).
            - Apply rotation formula:
                - If a function is rotated by $\theta$ degrees in the counter-clockwise direction then its new coordinates are given as follows: $$\begin{bmatrix}x'\\y'\end{bmatrix}= \begin{bmatrix}\cos(\theta)&-\sin(\theta)\\\sin(\theta)&\cos(\theta)\end{bmatrix}. \begin{bmatrix} x \\ y \end{bmatrix}$$
                - In our case, $\theta = 90^{\circ}$.
                    - The equation will become: $$y'=-x'+f(-x'-1)$$
        **Summary:** Thus the transformed function is $y=f^{-1}(−x+2)$ which makes option C correct.

In [9]:
display(Markdown(rft_base_eval_data.iloc[0]['parsed_answer']))

Thus, the transformed curve has an equation of $y = f^{−1} ( − x + 2 )$, making answer choice \boxed{C}

In [10]:
#Next 2 lines are required if running the code in Lightning AI Notebook:
import nest_asyncio
nest_asyncio.apply()

def get_rag_evaluation(embed_name, dataset, model_name):
    
    
    ragas_data = []
    for _, row in dataset.iterrows():
        
        obj ={
            'question' : row['question'],
            'contexts' : [row['contexts']],
            'ground_truth' : row['ground_truth'],
            'answer' : row['thought']
            
        }
        ragas_data.append(obj)
    
    data = Dataset.from_list(ragas_data)

    # 1. Setup an OpenAI-compatible client for Ollama
    # Ollama provides an OpenAI-compatible endpoint at /v1 --> If any other model to be used here:
    client = OpenAI()

    # 2. Use the llm_factory instead of LangchainLLMWrapper
    # Use provider="openai" to use the compatible client
    #LLM given here is the judge. So, use the same LLM that is used to generate this synthetic data
    evaluator_llm = llm_factory(model_name, client=client, max_tokens = 4096)

    #Throwing some errors so switched to LangchainEmbeddingsWrapper:
    # evaluator_embeddings = embedding_factory("openai", model=embed_name, client=client)

    # lc_embeddings = OllamaEmbeddings(
    # model=embed_name  # e.g. "nomic-embed-text"
    # )

    lc_embeddings = OpenAIEmbeddings(model=embed_name)
    ragas_embeddings = LangchainEmbeddingsWrapper(lc_embeddings)
    
    #Defining the metrics
    m1 = Faithfulness(llm=evaluator_llm)
    m2 = AnswerRelevancy(llm=evaluator_llm, embeddings=ragas_embeddings)
    m3 = AnswerCorrectness(llm=evaluator_llm, embeddings=ragas_embeddings)
    m4 = ContextPrecision(llm=evaluator_llm)

    metric_list = [m1, m2, m3, m4]
    
    results = evaluate(dataset=data,
                       metrics=metric_list,
                       embeddings=ragas_embeddings,
                        )
    eval_df = results.to_pandas()
    eval_df.to_csv('../data/rft_llama32-8bn_RFT_evaluationScores_v2.csv', index=False)
    return eval_df

In [11]:
eval_df = get_rag_evaluation(embed_name="text-embedding-ada-002", dataset=rft_base_eval_data, 
                             model_name="gpt-4o-mini") #Using the same model that we used earlier to generate the synthetic data

/tmp/ipykernel_116749/2779519256.py:38: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-openai package and should be used instead. To use it run `pip install -U `langchain-openai` and import as `from `langchain_openai import OpenAIEmbeddings``.
  lc_embeddings = OpenAIEmbeddings(model=embed_name)
/tmp/ipykernel_116749/2779519256.py:39: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(lc_embeddings)


Evaluating:   0%|          | 0/192 [00:00<?, ?it/s]

[ragas.prompt.pydantic_prompt|WARNING]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
[ragas.prompt.pydantic_prompt|WARNING]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
[ragas.prompt.pydantic_prompt|WARNING]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
[ragas.prompt.pydantic_prompt|WARNING]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
[ragas.prompt.pydantic_prompt|WARNING]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
[ragas.prompt.pydantic_prompt|WARNING]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
[ragas.prompt.pydantic_prompt|WARNING]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
[ragas.prompt.pydantic_prompt|WARNING]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
[ragas.prompt.pydantic_prompt|WARNING]LL

In [12]:
eval_df.head()

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,answer_correctness,context_precision
0,Return your final response within \boxed{}. Gi...,[['**Solution:**\nTo determine the equation of...,**Analysis:** The goal is to find the equation...,**Solution:**\nTo determine the equation of th...,0.384615,0.888615,0.399398,1.0
1,Return your final response within \boxed{}. Pr...,[['**Solution:**\n\nTo prove the inequality:\n...,The problem is moderately complex. I will brea...,**Solution:**\n\nTo prove the inequality:\n\[\...,0.642857,0.000000,0.604280,1.0
2,Return your final response within \boxed{}. If...,[['**Solution:**\nGiven the equation \\(\\log_...,Analysis: The problem is moderately complex. I...,**Solution:**\nGiven the equation \(\log_2 x^2...,0.250000,0.785849,0.361834,1.0
3,Return your final response within \boxed{}. A ...,[['**Solution:**\n1. The original ratio of but...,**Analysis:** The problem requires us to find ...,**Solution:**\n1. The original ratio of butter...,0.833333,0.846045,0.548842,1.0
4,Return your final response within \boxed{}. Gi...,[['**Solution:**\n\n1. **Separate the integer ...,The problem is moderately complex. I will brea...,**Solution:**\n\n1. **Separate the integer par...,0.307692,0.734672,0.630981,1.0


In [13]:
print("Metric Scores AFTER Reinforcement Fine-Tuning...!")
mean_faithfulness_score = eval_df['faithfulness'].mean()
mean_answer_relevancy_score = eval_df['answer_relevancy'].mean()
mean_answer_correctness_score = eval_df['answer_correctness'].mean()
print("Mean Faithfulness Score : ", mean_faithfulness_score)
print("Mean Answer Relevancy Score :", mean_answer_relevancy_score)
print("Mean Answer Correctness Score :", mean_answer_correctness_score)

Metric Scores AFTER Reinforcement Fine-Tuning...!
Mean Faithfulness Score :  0.3653131616495104
Mean Answer Relevancy Score : 0.6223555723666726
Mean Answer Correctness Score : 0.4811370001109985


In [ ]:
eval_df.to_csv('../data/rft_llama32-8bn_RFT_evaluationScores_v2.csv', index=False)

Checking model answers for some custom questions:

In [9]:
def prompt_formatting(prompt):
    text = (
            "<|begin_of_text|>"
            "<|start_header_id|>system<|end_header_id|>\n"
            "You are a mathematical reasoning assistant. Solve the problem step by step. Return your response EXACTLY  in this format:\n" 
            "<think>"
            "step-by-step reasoning"
            "</think>"
            "<answer>"
            "final answer"
            "</answer>"
            "<|eot_id|>"
            "<|start_header_id|>user<|end_header_id|>\n"
            f"{prompt}\n"
            "<|eot_id|>"
            "<|start_header_id|>assistant<|end_header_id|>\n"

        )

    return text

prompt = PromptTemplate(
                            template=RFT_EVAL_DATA_GEN_CONCISE,
                            input_variables = ["problem"]
                            )


rft_chain_eval = prompt | ft_model                            

In [21]:
text =prompt_formatting("""Lena buys 4 boxes of pencils. Each box contains 12 pencils. She gives 10 pencils to her friend and uses 8 pencils herself.
How many pencils does Lena have left?""")

response = rft_chain_eval.invoke(input={'problem':text})

print(response.content)

    <think>
        **Analysis:** The problem is simple, so I will use a direct solution method.

        **Approach:**
            - Calculate the number of pencils in each box (12).
            - Multiply by the total boxes bought (4) to get 48.
            - Subtract the amount given away and used (10 + 8 = 18), leaving 30 left over.
    </think>
    <answer>There are 30 pencils remaining.</answer>




RIGHT ANSWER ..! :D

In [20]:
text =prompt_formatting("""A farmer packs apples into crates. Each crate holds 25 apples. If the farmer fills 18 crates and then sells 200 apples,
how many apples remain?""")

response = rft_chain_eval.invoke(input={'problem':text})

print(response.content)

    <think>
        Analysis: The problem involves counting, so we can use a direct approach.
        
        Approach:
            - Count how many apples are in each full crate (25).
            - Multiply by the number of filled crates (18) to get 450 total apples.
            - Subtract from this the sold amount (200), leaving us with 250 remaining.

    </think>
    <answer>There remain \boxed{250} apples.</answer>




RIGHT ANSWER ..! :D

In [36]:
text =prompt_formatting("""Kevin reads 15 pages of a book every day for 6 days. After that, he reads 20 pages per day for 4 more days.
How many pages did Kevin read in total?""")

response = rft_chain_eval.invoke(input={'problem':text})

print(response.content)

    <think>
        **Analysis:** The problem involves counting the number of pages Kevin has read over several days with different page counts each day.

        **Approach:**
            - Calculate the sum of all daily readings
                15 + 20 = 35 (pages)
            - Multiply by total reading days to get overall count:
                6 * 35 = 210

    </think>
    <answer>Kevin read a total of \boxed{210} pages.</answer>




WRONG ANSWER ..! :(

In [40]:
text =prompt_formatting("""Sarah has 240 beads. She uses 36 beads to make bracelets and then divides the remaining beads equally into 6 jars.
How many beads go into each jar?""")

response = rft_chain_eval.invoke(input={'problem':text})

print(response.content)

    <think>
        **Analysis:** Sarah starts with 240 beads, makes 8 bracelets using 36 beads per bracelet (total of 288), leaving her with 48 leftover. She puts these in six equal-sized jars for a total of eight beads per jar.
        
        **Approach:**
            - Translate the problem into mathematical terms
                - Let x be the number of beads Sarah has left over after making bracelets
        
            - Apply arithmetic operations to find how many beads are left over:
            
                240 − (8 × 36) = 48
            
            - Relabel and simplify:
                
                240 − 288 = 48
                
        **Summary:** The answer is eight.
    </think>
    
    <answer>There are 8 beads in each jar.</answer>




WRONG ANSWER ..! :( Worse is model is hallucinating :O 

In [24]:
text =prompt_formatting("""A bookstore sells 18 books in the morning and twice as many books in the afternoon. Each book costs $12.
How much money did the bookstore make from selling the books that day?""")

response = rft_chain_eval.invoke(input={'problem':text})

print(response.content)

    <think>
        **Analysis:** The problem is moderately complex because it involves multiple steps to calculate total revenue, including determining how many books were sold each time and calculating their cost.

        **Initial Approach:**
            - Determine number of morning sales
                * 18 books in the morning
            - Calculate afternoon book count (twice as much)
                * 36 books in the afternoon

        **Reflection:** The initial approach is straightforward, but it's important to consider potential errors or assumptions that could impact accuracy.

        **Improvement:**
            - Double-check calculations for both sales periods.
            - Ensure all units are consistent throughout. (e.g., use "books" instead of "copies")

        **Summary:** By following these steps and double checking our work, we can confidently calculate the bookstore's total revenue from book sales that day.

    </think>
    <answer>

        The goal is to 

In [25]:
text =prompt_formatting("""A farmer harvested 45 baskets of tomatoes. Each basket contains 16 tomatoes. He sells 500 tomatoes at the market and keeps the rest.
How many tomatoes does the farmer keep?""")

response = rft_chain_eval.invoke(input={'problem':text})

print(response.content)

    <think>
        Analysis: The problem is moderately complex because it involves multiple steps to calculate how much he sold, then subtract that from his total harvest.

        Initial Approach:
            - Calculate number of baskets harvested
                45 x 16 = 720 (total tomatoes)
            - Determine amount kept by calculating the difference between total and what was sold.
                500 + ? = 720

        Reflection: The approach is not ideal because it requires manual calculation, which can be prone to error.

        Improvement:
            - Use a formula for number of baskets harvested
                n x m = t (n=number of baskets; m=num tomatoes per basket)
                    45x16=720
            - Calculate amount kept by subtracting sold from total.
                Total - Sold = Kept

    </think>
    <answer>
        The farmer keeps \boxed{220} tomatoes. To calculate, we use the formula for number of baskets harvested:
            n x m = t (n=

In [26]:
text =prompt_formatting("""Maya buys 7 packs of colored pencils. Each pack has 24 pencils. She gives 36 pencils to her classmates and uses 18 pencils for a project.
How many pencils does Maya have left?""")

response = rft_chain_eval.invoke(input={'problem':text})

print(response.content)

    <think>
        Analysis: The problem is moderately complex because it involves multiple steps, including counting the number of pencils given away and subtracting that from the total amount. It requires careful attention to detail.

        Initial Approach:
            - Count the number of packs she bought (7)
            - Multiply by 24 to get a total of 168
            - Subtract the number of pencils used for her project (18) 
            - Then, divide this result by two since each pack has 12 pencils

        Reflection: This approach is not efficient because it requires multiple steps and can be prone to errors. A more direct method would involve dividing directly.

        Improvement:
            - Divide the total number of packs bought (7) by half
            - Multiply that value by 24, which gives us a final answer of 84 pencils

    </think>
    <answer>Maya has 84 pencils left.</answer>




In [27]:
text =prompt_formatting("""A runner jogs 3 miles every morning and 5 miles every evening for 6 days. On the seventh day, the runner rests and does not run.
How many miles did the runner jog during the week?""")

response = rft_chain_eval.invoke(input={'problem':text})

print(response.content)

    <think>
        **Analysis:** The problem involves a repetitive pattern of jogging on certain days followed by rest.

        **Approach:**
            - Calculate total distance for each type (morning/evening) and sum them up.
            - Use arithmetic to simplify calculations, e.g., 3x6 + 5x6 = 18+30=48 miles jogged in the week

    </think>
    <answer>Thus, the runner jogged a total of \boxed{48} miles during the week.</answer>




In [28]:
text =prompt_formatting("""A warehouse receives 32 boxes of toys. Each box contains 25 toys. The warehouse ships 600 toys to different stores.
How many toys remain in the warehouse?""")

response = rft_chain_eval.invoke(input={'problem':text})

print(response.content)

    <think>
        Analysis: We have a total number of toys (N) and we know that each box has 25 toys, so N = 32 * 25
        Approach:
            - Calculate how many boxes are shipped to different stores. 
                - Each store receives the same amount of toys.
                - The warehouse ships 600 toys in total.
            - Subtract this number from the original number of toys (N) to get the remaining toys.

    </think>
    <answer>There are 200 toys left.</answer>




In [ ]:
text =prompt_formatting("""Kevin reads 15 pages of a book every day for 6 days. After that, he reads 20 pages per day for 4 more days.
How many pages did Kevin read in total?""")

response = rft_chain_eval.invoke(input={'problem':text})

print(response.content)

In [9]:
text =prompt_formatting("Susie has $200 in her piggy bank. If she puts 20% more money into her piggy bank, how much money she will have?")
response = rft_chain_eval.invoke(input={'problem':text})

print(response.content)

    <think>
        **Analysis:** Susie's current balance is $200.
        She wants to increase it by 20%.
        The new amount can be calculated as:
            - Original Balance: $200
            - Increase Amount (20%): ($200 * 0.2) = $40
            - New Total: $240

    </think>
    <answer>
        Thus, the transformed equation is:

            $$\begin{align*}
                \text{Original balance} &= \$200 \\
                \text{Increase amount (20%)} &=(\$200 * 0.2) = \$40\\
                \text{New total}&=\$240
            \end{align*}$$

        making the correct answer $\boxed{\color{red}{\mathbf{240}}}$.
    </answer>



RIGHT ANSWER ..! :D

In [10]:

text =prompt_formatting("""
                        Helen and Ivan had the same number of coins. Helen had a number of 50-cent coins, and 64 20-cent coins. These coins had a mass of 1.134kg. Ivan had a number of 50-cent coins and 104 20-cent coins.

                            (a) Who has more money in coins and by how much?
                            (b) given that each 50-cent coin is 2.7g more heavier than a 20-cent coin, what is the mass of Ivan's coins in kilograms?
                    """)
response = rft_chain_eval.invoke(input={'problem':text})

print(response.content)


                        <think>
    Analysis:
        - Helen had $x$ number of 50 cent and y number of 20 cents.
        - Ivan also has x number of 50 cent and z number of 20 cents
        - The total weight is given as 1.134kg, so we can calculate the mass of each coin using this information.

    Approach:
        - We will use algebra to solve for $x$ and $y$
        
    Summary: 
        - Helen has more money than Ivan by a certain amount.
        - The mass of Ivan's coins is given as 1.134kg
</think>
<answer>


WRONG ANSWER ..! :( BUT VERY VERY CLOSE, SO :) 

In [11]:


text =prompt_formatting("""
                            Jess needs 200 pieces of ribbon, each of length 110cm, to decorate a room for a party.
                            Ribbon is sold in rolls of 25m each.

                            What is the least number of rolls of ribbon that Jess needs to buy?
                        """)
response = rft_chain_eval.invoke(input={'problem':text})

print(response.content)


    <think>
        Analysis: The problem requires finding the minimum number of rolls needed to cover an area. We can use division and multiplication to calculate this.
        
        Approach:
            - Calculate how many meters are required for 200 pieces (110cm each): 22000 cm / 100 = 220 m
            - Divide by roll length: 220m/25m=8 rolls
        
    </think>
    <answer>Jess needs at least 8 rolls of ribbon.</answer>




WRONG ANSWER ..! :(

In [12]:

text =prompt_formatting( """
                            Could George Washington have had dinner with Barack Obama? Work out the answer step by step.
                        """)
response = rft_chain_eval.invoke(input={'problem':text})

print(response.content)

    <think>
        The problem is moderately complex. I will break it down into key points and challenges, then evaluate preliminary solutions before refining them.

        **Analysis:**
            - This question requires understanding of historical events and timelines
            - We need to determine if George Washington was alive during Barack Obama's presidency
        
        **Initial Approach:** 
            - Check the dates when both men were in office. If they overlapped, then it is possible for a dinner meeting.
        
        **Reflection:**
            - The initial approach seems reasonable but may not be sufficient as there could have been other factors that prevented them from having dinner together.

        **Improvement:** 
            - We can refine the solution by considering additional information such as their political affiliations, personal relationships and social circles. This will help us determine if they had any common acquaintances or shared int

RIGHT ANSWER ..! :D

In [13]:


text =prompt_formatting( """
                            Four people lost a total of 103 kilograms of weight. 
                            The first person lost 27 kilograms. 
                            The second person lost 7 kilograms less than the first person. 
                            The two remaining people lost the same amount. 
                            How many kilograms did each of the last two people lose?
                    """)
response = rft_chain_eval.invoke(input={'problem':text})

print(response.content)

    <think>
        Analysis: Four people, total weight loss is 103 kg.
        
        Approach:
            - First Person (27)
            - Second person (27-7=20)    
            - Third and Fourth persons are same
                - Let x be the amount lost by third person. 
                    Then fourth person also loses 'x' kilograms.

    </think>
    <answer>Third and forth people lose 28 kg each.</answer>




RIGHT ANSWER ..! :D

In [29]:
text =prompt_formatting("""A teacher buys 6 packs of markers. Each pack contains 14 markers. She gives 18 markers to another teacher and then distributes the rest equally among 8 students.
How many markers does each student receive? """)
response = rft_chain_eval.invoke(input={'problem':text})

print(response.content)

    <think>
        Analysis: The problem involves dividing a total number (number of markers) by two numbers, one for teachers and other for students.

        Approach:
            - Calculate the total number of markers
            - Subtract the 18 given to another teacher from this value.
            - Divide the result by 8 to get the amount each student receives. 
    </think>
    <answer>Each student gets \boxed{3} markers.</answer>




In [10]:
text =prompt_formatting("""A delivery truck carries 120 boxes of oranges. At the first store, it delivers one-third of the boxes. At the second store, it delivers 25 boxes.
How many boxes remain on the truck? """)
response = rft_chain_eval.invoke(input={'problem':text})

print(response.content)

    <think>
        **Analysis:** The problem involves a starting quantity (boxes), two deliveries that reduce this amount by different amounts, and an unknown final count.

        **Approach:**
            - Calculate the number of boxes delivered at each store.
            - Subtract these from 120 to find remaining boxes. 

        **Summary**: 
            We will calculate how many boxes are left after delivering one-third (40) at the first stop and then another 25 at the second, leaving us with a total of 55 boxes on the truck.

    </think>
    <answer>There are \boxed{55} boxes remaining.</answer>




In [11]:
text =prompt_formatting("""A library has 360 books on three shelves. The first shelf has 120 books. The second shelf has 40 more books than the first shelf.
How many books are on the third shelf?""")
response = rft_chain_eval.invoke(input={'problem':text})

print(response.content)

    <think>
        **Analysis:** We have a total of 360 books, with 120 in the first shelf and an unknown number (x) in the second shelf.

        **Approach:**
            - Let x be the number of books in the second shelf.
            - The third shelf has y = 360 – (120 + x) books. 
    </think>
    <answer>Thus, there are \boxed{180} books on the third shelf.</answer>




In [12]:
text =prompt_formatting("""A factory produces 75 toys each day. After producing toys for 8 days, the factory ships 420 toys to stores.
How many toys remain in the factory?""")
response = rft_chain_eval.invoke(input={'problem':text})

print(response.content)

    <think>
        **Analysis:** The problem involves a simple calculation of remaining items after some are shipped.

        **Approach:**
            - Calculate total number produced (75 x 8 = 600)
            - Subtract shipments sent (420) from production
                - Remaining toys in factory is 180.
        
    </think>
    <answer>There remain \boxed{180} toys.</answer>




In [13]:
text =prompt_formatting("""There are 240 students in a school. One-fourth of the students are in grade 6, and one-third of the remaining students are in grade 7.
How many students are in grade 7?""")
response = rft_chain_eval.invoke(input={'problem':text})

print(response.content)

    <think>
        **Analysis:** The problem involves counting students by grades.

        **Approach:**
            - Calculate total number of students
            - Determine fraction of Grade 6 and Grade 7 students
            - Multiply fractions to find the number of Grade 7 students.
        
        **Summary**: We will calculate the total number of students, determine their grade distribution (one-fourth in Grade 6), then use this information to compute how many are in Grade 7. 
    </think>
    <answer>There are \boxed{80} students in Grade 7.</answer>




STRESS TESTING

Hidden subtraction trap

In [24]:
text =prompt_formatting("""James has 120 marbles. He gives 25 marbles to his friend and then buys 40 more marbles from a store. Later he loses 15 marbles while playing.
How many marbles does James have now?""")
response = rft_chain_eval.invoke(input={'problem':text})

print(response.content)

    <think>
        Analysis: The problem involves addition, subtraction, multiplication, division, and inequality operations.

        Approach:
            - Start with the initial number of marbles (120).
            - Subtract the amount given to a friend (25) from this total.
            - Add the new marbles purchased (40)
            - Then subtract any lost marbles (15)

    </think>
    <answer>

        James has 140 marbles now.

    </answer>


WRONG ANSWER..! :(

In [23]:
text =prompt_formatting("""James has 123 marbles. He gives 25 marbles to his friend and then buys 40 more marbles from a store. Later he loses 15 marbles while playing.
How many marbles does James have now?""")
response = rft_chain_eval.invoke(input={'problem':text})

print(response.content)

    <think>
        Analysis: The problem involves addition, subtraction, multiplication, division, and inequality operations.

        Approach:
            - Start with the initial number of marbles (123).
            - Subtract the given amount to his friend (25) from this.
            - Add 40 for buying more marbles.
            - Then subtract lost marbles (15).

    </think>
    <answer>James has \boxed{103} marbles now.</answer>




WRONG ANSWER..! :(

Fraction reasoning trap

In [19]:
text =prompt_formatting("""A classroom has 36 students. Two-thirds of the students bring lunch from home. The rest buy lunch from the cafeteria.

How many students buy lunch from the cafeteria?""")
response = rft_chain_eval.invoke(input={'problem':text})

print(response.content)

    <think>
        **Analysis:** There are two groups: those who brought their own lunches and those who bought theirs at school.
        
        **Approach:** We can use a formula to calculate how many people in each group. Then, we subtract one number from another to find the answer.

        **Summary:** The total is 36 students. Two-thirds of them bring lunch from home (24), so that leaves two-thirds who buy their lunches at school.
    </think>
    <answer>12</answer>




RIGHT ANSWER ..! :D

Order-of-operations trap

In [16]:
text =prompt_formatting("""A bakery makes 24 cakes every hour. After 5 hours, they sell 60 cakes and give away 12 cakes for free.
How many cakes remain?""")
response = rft_chain_eval.invoke(input={'problem':text})

print(response.content)

    <think>
        **Analysis:** The problem involves a simple calculation of the number of remaining cakes after selling some and giving others away.

        **Approach:**
            - Calculate the total number of sold + given-away cakes (60+12=72).
            - Subtract this from 24*5 to get the remaining cakes.
    </think>
    <answer>There are \boxed{48} cakes left.</answer>




RIGHT ANSWER..! :D 

Multi-step proportional reasoning

In [34]:
text =prompt_formatting("""A farmer has 90 chickens. One-third of the chickens lay eggs every day. Each of those chickens lays 2 eggs.

How many eggs are produced in one day?""")
response = rft_chain_eval.invoke(input={'problem':text})

print(response.content)

    <think>
        The problem is moderately complex, so I will use a structured approach:
        
        **Analysis:** We have to find how many eggs each chicken produces per day and multiply it by the number of chickens that lay eggs. Then we add up all those numbers.
        
        **Initial Approach:** Let's assume there are 30 chickens laying eggs every day (one-third of total). Each one lays two eggs, so they produce a total of 60 eggs in one day.

        **Reflection:** This seems like the right approach but let me check if it works for other cases too.
        
        **Improvement:** I can use this formula: number_of_eggs = (number_of_chickens * egg_production_rate) / daily_egg_yield
        
        So, 90 chickens x 2 eggs per chicken ÷ 3 days gives us the total amount of eggs produced in one day which is equal to 60.
        
    </think>
    
    <answer>There are a total of 60 eggs laid by these chickens every single day.</answer>




RIGHT ANSWER ..! :D 

Sequential operations trap

In [35]:
text =prompt_formatting("""A store receives 200 notebooks. On the first day it sells half of them. On the second day it sells 30 notebooks. On the third day it receives 40 more notebooks.

How many notebooks does the store have now?""")
response = rft_chain_eval.invoke(input={'problem':text})

print(response.content)

    <think>
        **Analysis:** The problem involves a sequence of events, with each event affecting the number of remaining notebooks.
        
        **Approach:**
            - Define variables for initial stock and daily sales
            - Use an equation to represent the change in notebook count after each day
        
        **Summary:** We will use algebraic equations to model the changes in notebook counts over time. The final solution involves finding a formula that relates the number of notebooks remaining at any point.
    </think>
    
    <answer>
        Let $n$ be the initial stock, and let $s_1$, $s_2$, and $s_3$ represent sales on days 1-3 respectively. We have:
        
        $$\begin{align*}
            n &= \text{(initial notebooks)} \\
            s_1 &= \frac{n}{2} = \text{(notebooks sold day 1)}\\
            s_2 &= 30 = \text{(notebooks sold day 2)}\\
            s_3 &= ? = \text{(notebooks received on day 3)}
        \end{align*}
        
        After t

WRONG ANSWER ..! :(